In [ ]:
import random
from collections import deque

# Helper Functions
def manhattan(cell1, cell2): return abs(cell1[0] - cell2[0]) + abs(cell1[1] - cell2[1])

def get_neighbors(grid, row, col):
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    neighbors = []
    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc
        if (0 <= new_row < len(grid) and 0 <= new_col < len(grid[0]) and grid[new_row][new_col] == 1):
            neighbors.append((new_row, new_col))
    return neighbors

def bfs_reachable(grid, start, goal):
    queue = deque([start])
    visited = {start}
    while queue:
        r, c = queue.popleft()
        if (r, c) == goal:
            return True
        for neighbor in get_neighbors(grid, r, c):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return False

def minimax(grid, wizard, enemy, depth, is_maximizing, goal):
    """
    Adversarial search — Wizard (MAX) vs Enemy (MIN).
    Wizard tries to reach goal, Enemy tries to intercept.
    Returns: best score (int)
    """
    if depth == 0 or wizard == goal:
        return -manhattan(wizard, goal) + manhattan(enemy, wizard)
    
    if is_maximizing:
        best = float('-inf')
        for move in get_neighbors(grid, *wizard):
            val = minimax(grid, move, enemy, depth-1, False, goal)
            best = max(best, val)
        return best
    else:
        best = float('inf')
        for move in get_neighbors(grid, *enemy):
            val = minimax(grid, wizard, move, depth-1, True, goal)
            best = min(best, val)
        return best

def run_minimax_game(grid, start, goal, depth_limit=3):
    """
    Runs full Minimax game loop — wizard and enemy alternate turns.
    Enemy starts near center of grid.
    Returns: (wizard_path, enemy_path, result)
    """
    wizard = start
    enemy = (10, 12)  # enemy starts near center
    wizard_path = [wizard]
    enemy_path = [enemy]
    
    for turn in range(200):
        # Wizard's turn (MAX)
        neighbors = get_neighbors(grid, *wizard)
        if not neighbors:
            break
        best_move = max(neighbors, 
                        key=lambda m: minimax(grid, m, enemy, depth_limit-1, False, goal))
        wizard = best_move
        wizard_path.append(wizard)
        
        # Enemy's turn (MIN) — moves toward wizard
        e_neighbors = get_neighbors(grid, *enemy)
        if e_neighbors:
            enemy = min(e_neighbors, key=lambda m: manhattan(m, wizard))
        enemy_path.append(enemy)
        
        print(f"Minimax: Wizard moves to ({wizard[0]}, {wizard[1]}); Enemy moves to ({enemy[0]}, {enemy[1]})")
        
        if wizard == goal:
            return wizard_path, enemy_path, "Wizard reached goal!"
        if wizard == enemy:
            return wizard_path, enemy_path, "Enemy caught wizard!"
    
    return wizard_path, enemy_path, "Draw"

def csp_generate_dungeon(rows=20, cols=24, start=(1,1), goal=(18,22)):
    """
    Constraint Satisfaction Problem — generates dungeon with backtracking.
    Constraint: a valid path from start to goal must always exist.
    Returns: grid (2D list)
    """
    # Initialize all walls
    grid = [[0]*cols for _ in range(rows)]
    
    # Step 1: Carve guaranteed path (zigzag corridor)
    r, c = start
    grid[r][c] = 1
    while (r, c) != goal:
        if c < goal[1]:
            c += 1
        elif r < goal[0]:
            r += 1
        grid[r][c] = 1
    
    # Step 2: Add random rooms
    for _ in range(6):
        rr = random.randint(1, rows-4)
        rc = random.randint(1, cols-4)
        for dr in range(random.randint(2,4)):
            for dc in range(random.randint(2,4)):
                if 0 < rr+dr < rows-1 and 0 < rc+dc < cols-1:
                    grid[rr+dr][rc+dc] = 1
    
    # Step 3: CSP wall placement with backtracking
    floor_cells = [(rh, ch) for rh in range(rows) for ch in range(cols) 
                   if grid[rh][ch]==1 and (rh, ch)!=start and (rh, ch)!=goal]
    random.shuffle(floor_cells)
    
    for (r, c) in floor_cells[:40]:  # try placing 40 extra walls
        grid[r][c] = 0
        if bfs_reachable(grid, start, goal):
            print(f"CSP: Wall placed at ({r}, {c}) — path still valid")
        else:
            grid[r][c] = 1
            print(f"CSP: Constraint violated at ({r}, {c}) — backtracking!")
    
    return grid

# Demo
g = csp_generate_dungeon()
run_minimax_game(g, (1, 1), (18, 22))